# 04 - Fine-Tuning (RQ3)

Does fine-tuning on labeled PropBank data beat prompting?

**Cost warning:** fine-tuning spends credits differently from inference.
Get Eric's approval and confirm the base model is fine-tuning eligible before running.

In [ ]:
import sys, os
from pathlib import Path
# Make src/ importable whether run from repo root or notebooks/.
ROOT = Path.cwd()
if (ROOT / 'src').exists():
    sys.path.insert(0, str(ROOT / 'src'))
elif (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
    sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)
print('project root:', ROOT)


In [ ]:
import data_loader as dl, srl_inference as si
client = si.get_client()
train = dl.read_jsonl('data/train.jsonl')
val = dl.read_jsonl('data/validation.jsonl')

## Export chat-format JSONL for fine-tuning

In [ ]:
n_tr = si.export_finetune_jsonl(train, 'data/ft_train.jsonl')
n_va = si.export_finetune_jsonl(val, 'data/ft_validation.jsonl')
print('wrote', n_tr, 'train and', n_va, 'validation records')
print(open('data/ft_train.jsonl').readline()[:300])

## Upload files and create the job
Uncomment to spend credits. The base model comes from `SRL_FT_BASE_MODEL`.

In [ ]:
# train_file = si.upload_file(client, 'data/ft_train.jsonl')
# val_file = si.upload_file(client, 'data/ft_validation.jsonl')
# job_id = si.create_finetune_job(client, train_file, validation_file_id=val_file,
#                                 base_model=si.DEFAULT_FT_BASE)
# print('job:', job_id)
# ft_model = si.poll_finetune_job(client, job_id)
# print('fine-tuned model:', ft_model)
# >>> Copy ft_model into .env as SRL_FINETUNED_MODEL, then restart kernel. <<<

## Evaluate the fine-tuned model
Run after `SRL_FINETUNED_MODEL` is set in `.env`.

In [ ]:
import evaluation as ev
test = dl.read_jsonl('data/test.jsonl')
ft_model = si.FINETUNED_MODEL
if not ft_model:
    print('SRL_FINETUNED_MODEL not set; skipping. Run the job above first.')
else:
    res = si.predict_batch(client, test, model=ft_model)
    si.save_predictions(res, 'results/predictions_fine_tuned.jsonl')
    m = ev.evaluate(res, test)
    print(ev.summary_row('fine-tuned', m))